# Tugas 1 - Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data
**Nama Mahasiswa:** Fatahillah Al-Fatih
**NIM:** 202310370311132
**Username GitHub:** FatihHDR

Sumber data: `data/processed/dataset_bersih.parquet` (hasil Milestone 2) - cuaca per jam untuk 40 kota di 38 provinsi Indonesia, 2001-2024 (8.415.360 baris). Notebook ini berisi analisis **temporal** (window function) dan **spasial**, **6 visualisasi interaktif Plotly**, serta **5 insight**.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat data hasil pembersihan dengan Polars *lazy* dan sambungkan DuckDB untuk agregasi SQL.

In [1]:
from pathlib import Path

import duckdb
import polars as pl
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'
FIG_DIR = PROJECT_ROOT / 'output' / 'figures'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Data bersih tidak ditemukan: {DATA_PATH}. Jalankan notebooks/02_data_cleaning.ipynb lebih dahulu.'
    )

FIG_DIR.mkdir(parents=True, exist_ok=True)
lazy_data = pl.scan_parquet(DATA_PATH)
connection = duckdb.connect()

ringkas = connection.execute(
    '''
    SELECT MIN(waktu_lokal) AS awal, MAX(waktu_lokal) AS akhir,
           COUNT(DISTINCT kota) AS kota, COUNT(DISTINCT provinsi) AS provinsi
    FROM read_parquet(?)
    ''',
    [str(DATA_PATH)],
).fetchall()[0]

print(f'Data bersih : {DATA_PATH.name} ({DATA_PATH.stat().st_size / 1_000_000:.1f} MB)')
print(f'Periode     : {ringkas[0]} s.d. {ringkas[1]}')
print(f'Cakupan     : {ringkas[2]} kota, {ringkas[3]} provinsi')

Data bersih : dataset_bersih.parquet (192.4 MB)
Periode     : 2001-01-01 00:00:00 s.d. 2024-12-31 23:00:00
Cakupan     : 40 kota, 38 provinsi


## Bagian 2: Analisis Temporal dan Window Functions

**a. Transformasi lazy Polars.** `group_by_dynamic` mengagregasi data per jam menjadi seri bulanan rata-rata nasional, lalu `rolling_mean` menghaluskan tren jangka panjang.

In [2]:
suhu_bulanan = (
    lazy_data.sort('waktu_lokal')
    .group_by_dynamic('waktu_lokal', every='1mo')
    .agg(pl.col('T2M').mean().alias('suhu_rata2'))
    .sort('waktu_lokal')
    .collect()
    .with_columns(
        pl.col('suhu_rata2').rolling_mean(window_size=12, min_samples=12).alias('ma12_polars')
    )
)
print(f'Deret bulanan: {suhu_bulanan.shape[0]} bulan')
suhu_bulanan.head()

Deret bulanan: 288 bulan


waktu_lokal,suhu_rata2,ma12_polars
datetime[μs],f64,f64
2001-01-01 00:00:00,25.166433,null
2001-02-01 00:00:00,25.24166,null
2001-03-01 00:00:00,25.399088,null
2001-04-01 00:00:00,25.78452,null
2001-05-01 00:00:00,25.905382,null


**b. Window function DuckDB (`OVER`).** Rata-rata bergerak 12 bulan dihitung ulang dengan `AVG(...) OVER (...)`, sekaligus memberi peringkat bulan terpanas sepanjang periode.

In [3]:
bulanan_duckdb = pl.from_arrow(
    connection.sql(
        '''
        WITH bulanan AS (
            SELECT date_trunc('month', waktu_lokal) AS bulan, AVG(T2M) AS suhu
            FROM read_parquet(?)
            GROUP BY 1
        )
        SELECT bulan, ROUND(suhu, 3) AS suhu_rata2,
               ROUND(AVG(suhu) OVER (
                   ORDER BY bulan ROWS BETWEEN 11 PRECEDING AND CURRENT ROW), 3
               ) AS ma12_duckdb,
               RANK() OVER (ORDER BY suhu DESC) AS peringkat_panas
        FROM bulanan
        ORDER BY bulan
        ''',
        params=[str(DATA_PATH)],
    ).to_arrow_table()
)
print('5 bulan terpanas:')
bulanan_duckdb.filter(pl.col('peringkat_panas') <= 5).sort('peringkat_panas')

5 bulan terpanas:


bulan,suhu_rata2,ma12_duckdb,peringkat_panas
datetime[μs],f64,f64,i64
2023-11-01 00:00:00,27.179,26.057,1
2023-10-01 00:00:00,27.115,25.952,2
2019-11-01 00:00:00,27.06,25.998,3
2024-04-01 00:00:00,27.003,26.458,4
2024-10-01 00:00:00,26.915,26.556,5


**c. Tren pemanasan per kota.** Regresi linear suhu tahunan terhadap waktu (`regr_slope`) mengukur laju perubahan suhu tiap kota dalam derajat Celsius per tahun.

In [4]:
tren_kota = pl.from_arrow(
    connection.sql(
        '''
        WITH tahunan AS (
            SELECT kota, provinsi, EXTRACT(year FROM waktu_lokal) AS tahun, AVG(T2M) AS suhu
            FROM read_parquet(?)
            GROUP BY 1, 2, 3
        )
        SELECT kota, provinsi,
               ROUND(regr_slope(suhu, tahun), 4) AS tren_c_per_tahun,
               ROUND(AVG(suhu), 2) AS suhu_rata2
        FROM tahunan
        GROUP BY kota, provinsi
        ORDER BY tren_c_per_tahun DESC
        ''',
        params=[str(DATA_PATH)],
    ).to_arrow_table()
)
print(f'Rata-rata laju pemanasan: {tren_kota["tren_c_per_tahun"].mean():.4f} derajat C/tahun')
tren_kota.head(10)

Rata-rata laju pemanasan: 0.0226 derajat C/tahun


kota,provinsi,tren_c_per_tahun,suhu_rata2
str,str,f64,f64
"""Tanjung Pinang""","""Kepulauan Riau""",0.0382,27.91
"""Mataram""","""Nusa Tenggara Barat""",0.0333,26.48
"""Sorong""","""Papua Barat Daya""",0.0332,27.35
"""Balikpapan""","""Kalimantan Timur""",0.0292,26.16
"""Medan""","""Sumatera Utara""",0.0291,25.84
"""Wamena""","""Papua Pegunungan""",0.0281,13.43
"""Denpasar""","""Bali""",0.028,25.14
"""Mamuju""","""Sulawesi Barat""",0.0278,27.84
"""Manado""","""Sulawesi Utara""",0.0277,26.73


## Bagian 3: Enam Visualisasi Interaktif (Plotly)

Data agregat disiapkan lebih dulu agar tiap figur hanya memuat ringkasan kecil (bukan 8,4 juta baris).

In [5]:
kolom_numerik = [
    'T2M', 'T2MDEW', 'RH2M', 'QV2M', 'PRECTOTCORR',
    'PS', 'WS10M', 'WD10M', 'ALLSKY_SFC_SW_DWN', 'CLOUD_AMT',
]

kota_ringkas = pl.from_arrow(
    connection.sql(
        '''
        SELECT kota, provinsi, zona_waktu, latitude, longitude,
               ROUND(AVG(T2M), 2) AS suhu_rata2,
               ROUND(AVG(RH2M), 1) AS rh_rata2,
               ROUND(SUM(PRECTOTCORR), 0) AS hujan_total_mm,
               ROUND(AVG(PRECTOTCORR) * 24, 2) AS hujan_mm_per_hari
        FROM read_parquet(?)
        GROUP BY 1, 2, 3, 4, 5
        ''',
        params=[str(DATA_PATH)],
    ).to_arrow_table()
)

suhu_bulanan_kota = pl.from_arrow(
    connection.sql(
        '''
        SELECT kota, provinsi, date_trunc('month', waktu_lokal) AS bulan, AVG(T2M) AS suhu
        FROM read_parquet(?)
        GROUP BY 1, 2, 3
        ''',
        params=[str(DATA_PATH)],
    ).to_arrow_table()
)

hujan_prov = pl.from_arrow(
    connection.sql(
        '''
        SELECT provinsi, ROUND(SUM(PRECTOTCORR), 0) AS hujan_total_mm
        FROM read_parquet(?)
        GROUP BY 1
        ''',
        params=[str(DATA_PATH)],
    ).to_arrow_table()
)

siklus = pl.from_arrow(
    connection.sql(
        '''
        SELECT EXTRACT(hour FROM waktu_lokal) AS jam, EXTRACT(month FROM waktu_lokal) AS bulan,
               AVG(T2M) AS suhu
        FROM read_parquet(?)
        GROUP BY 1, 2
        ''',
        params=[str(DATA_PATH)],
    ).to_arrow_table()
)
matriks_suhu = siklus.pivot(on='jam', index='bulan', values='suhu').sort('bulan').drop('bulan')

pair_sql = ', '.join(f'corr("{a}", "{b}") AS "{a}|{b}"' for a in kolom_numerik for b in kolom_numerik)
nilai_korelasi = connection.execute(
    f'SELECT {pair_sql} FROM read_parquet(?)', [str(DATA_PATH)]
).fetchall()[0]
matriks_korelasi = [
    [nilai_korelasi[i * len(kolom_numerik) + j] for j in range(len(kolom_numerik))]
    for i in range(len(kolom_numerik))
]

print('Agregat siap: kota_ringkas, suhu_bulanan_kota, hujan_prov, matriks_suhu, matriks_korelasi')

Agregat siap: kota_ringkas, suhu_bulanan_kota, hujan_prov, matriks_suhu, matriks_korelasi


**Visualisasi 1 - Tren deret waktu (line chart + range slider).** Suhu bulanan nasional beserta rata-rata bergerak 12 bulan.

In [6]:
fig1 = px.line(
    suhu_bulanan, x='waktu_lokal', y=['suhu_rata2', 'ma12_polars'],
    labels={'waktu_lokal': 'Bulan', 'value': 'Suhu (derajat C)', 'variable': 'Deret'},
    title='1. Tren Suhu Bulanan Nasional (rata-rata 40 kota), 2001-2024',
)
fig1.data[0].name = 'Suhu bulanan'
fig1.data[1].name = 'Rata-rata bergerak 12 bulan'
fig1.update_xaxes(rangeslider_visible=True)
fig1.show()

**Visualisasi 2 - Distribusi variabel utama (box plot).** Sebaran suhu bulanan tiap kota, diurutkan berdasarkan median.

In [7]:
urutan_kota = (
    suhu_bulanan_kota.group_by('kota').agg(pl.col('suhu').median().alias('median'))
    .sort('median')['kota'].to_list()
)
fig2 = px.box(
    suhu_bulanan_kota, x='kota', y='suhu', color='provinsi',
    category_orders={'kota': urutan_kota}, points=False,
    labels={'kota': 'Kota', 'suhu': 'Suhu bulanan (derajat C)', 'provinsi': 'Provinsi'},
    title='2. Sebaran Suhu Bulanan per Kota (2001-2024)',
)
fig2.update_xaxes(tickangle=90)
fig2.update_layout(showlegend=False)
fig2.show()

**Visualisasi 3 - Korelasi antar variabel (heatmap).** Matriks korelasi Pearson 10 variabel cuaca.

In [8]:
fig3 = px.imshow(
    matriks_korelasi, x=kolom_numerik, y=kolom_numerik, zmin=-1, zmax=1,
    color_continuous_scale='RdBu_r', text_auto='.2f', aspect='auto',
    title='3. Matriks Korelasi Antar Variabel Cuaca',
)
fig3.update_layout(height=640)
fig3.show()

**Visualisasi 4 - Komposisi kategori (treemap).** Pangsa total curah hujan tiap provinsi sepanjang periode.

In [9]:
fig4 = px.treemap(
    hujan_prov, path=['provinsi'], values='hujan_total_mm',
    color='hujan_total_mm', color_continuous_scale='Blues',
    title='4. Komposisi Total Curah Hujan per Provinsi (2001-2024)',
)
fig4.show()

**Visualisasi 5 - Siklus musiman dan harian (density heatmap).** Suhu rata-rata per jam lokal dan per bulan, memperlihatkan pola harian sekaligus pola musiman.

In [10]:
label_bulan = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']
fig5 = px.imshow(
    matriks_suhu.to_numpy(), x=[f'{jam:02d}' for jam in range(24)], y=label_bulan,
    aspect='auto', color_continuous_scale='Turbo',
    labels=dict(x='Jam lokal', y='Bulan', color='Suhu (derajat C)'),
    title='5. Siklus Harian dan Musiman Suhu (jam x bulan)',
)
fig5.show()

**Visualisasi 6 - Pola spasial (peta).** Sebaran geografis kota: warna menunjukkan suhu rata-rata, ukuran menunjukkan total curah hujan.

In [11]:
fig6 = px.scatter_geo(
    kota_ringkas, lat='latitude', lon='longitude', color='suhu_rata2',
    size='hujan_total_mm', hover_name='kota',
    hover_data=['provinsi', 'zona_waktu', 'hujan_mm_per_hari'],
    scope='asia', color_continuous_scale='Inferno',
    labels={'suhu_rata2': 'Suhu rata-rata (derajat C)'},
    title='6. Pola Spasial Suhu (warna) dan Total Curah Hujan (ukuran) per Kota',
)
fig6.update_geos(showcountries=True, countrycolor='lightgray', landcolor='whitesmoke')
fig6.show()

Simpan seluruh figur sebagai berkas HTML interaktif mandiri di `output/figures/`.

In [12]:
figures = {
    '01_tren_suhu_bulanan': fig1,
    '02_sebaran_suhu_kota': fig2,
    '03_matriks_korelasi': fig3,
    '04_treemap_hujan_provinsi': fig4,
    '05_siklus_jam_bulan': fig5,
    '06_pola_spasial': fig6,
}
for nama, figur in figures.items():
    figur.write_html(FIG_DIR / f'{nama}.html', include_plotlyjs='cdn')

print(f'{len(figures)} figur interaktif disimpan di {FIG_DIR}')

6 figur interaktif disimpan di /home/fatih/Documents/Projects/tugas1-abd-2026/output/figures


## Bagian 4: Rumusan Lima Insight Analitik

**1. Tren pemanasan jangka panjang dan bulan terpanas (temporal).**
Suhu nasional menghangat sekitar **+0,023 derajat C/tahun**; rata-rata periode 2001-2005 (25,70 derajat C) naik menjadi 26,05 derajat C pada 2020-2024 (**+0,35 derajat C**). Seluruh **40 kota** menunjukkan tren positif, tercepat di Tanjung Pinang (+0,038 derajat C/tahun), Mataram, dan Sorong. Empat dari lima bulan terpanas sepanjang periode terjadi pada 2023-2024 (Oktober-November 2023 memuncak di 27,18 derajat C), konsisten dengan El Nino kuat 2023-2024.

**2. Ketimpangan suhu antar wilayah (spasial).**
Terdapat jurang suhu yang besar antar kota: Wamena (Papua Pegunungan) hanya **13,43 derajat C**, sementara Tanjung Pinang **27,91 derajat C** - selisih **~14,5 derajat C**. Perbedaan ini didorong faktor ketinggian (Wamena ~1.600 m; Bandung juga relatif sejuk, 21,49 derajat C), bukan posisi lintang. Peta menegaskan klaster dingin di dataran tinggi Papua dan panas di pesisir/kepulauan.

**3. Korelasi kunci antar variabel.**
Korelasi terkuat adalah `T2M`-`PS` (**+0,74**) dan `T2M`-`RH2M` (**-0,64**), diikuti `T2M`-`ALLSKY_SFC_SW_DWN` (+0,45) dan `T2M`-`CLOUD_AMT` (-0,13). Interpretasi perlu hati-hati: korelasi suhu-tekanan yang tinggi sebagian besar berasal dari perbedaan **antar-lokasi** (ketinggian), bukan hubungan sebab-akibat - contoh klasik *confounding* spasial.

**4. Pola musiman dan siklus harian.**
Curah hujan sangat kontras secara musiman: Desember **8,80 mm/hari** versus Agustus **3,81 mm/hari** (rasio **2,3x**), menandai musim hujan Desember-Februari dan kemarau Juli-Agustus. Siklus harian menunjukkan suhu memuncak pukul **13.00 (28,35 derajat C)** dan terendah pukul **05.00 (23,83 derajat C)** dengan rentang **~4,5 derajat C**. Kota terbasah: Nabire (10,44 mm/hari), Padang, dan Pontianak.

**5. Rekomendasi praktis.**
(a) **Pertanian:** selaraskan kalender tanam dengan puncak hujan Desember-Januari dan siapkan irigasi untuk kemarau Juli-Agustus. (b) **Kesehatan & perkotaan:** pantau risiko *heat stress* pada Oktober-November, terutama di kota pesisir panas dengan kelembapan tinggi. (c) **Adaptasi iklim:** jadikan laju **+0,023 derajat C/tahun** sebagai baseline perencanaan dan prioritaskan kota dengan pemanasan tercepat. (d) **Energi:** korelasi positif suhu-radiasi matahari mendukung potensi PLTS. (e) **Analitik:** pisahkan efek antar-lokasi dan dalam-lokasi sebelum menarik kesimpulan sebab-akibat dari korelasi.